In [2]:
import os
import math
import pandas as pd
import spikeinterface.full as si
# read the matlab files
import numpy as np
import scipy.io
import pandas as pd

def read_probe_mat(probe_locs_path):
    mat = scipy.io.loadmat(probe_locs_path)
    probe_locs = np.array(mat['probe_locs'])
    print(probe_locs)
    return probe_locs
 
def read_borders_table(border_tables_path):
    mat = scipy.io.loadmat(border_tables_path)
    borders_table = pd.DataFrame(mat['borders_table'])
    print(borders_table)
    return borders_table

In [5]:
# Function to convert stereotaxic coordinates to ABA CCF
# SC is an array with stereotaxic coordinates to be transformed
# Returns an array containing corresponding CCF coordinates in μm
# Conversion is from this post, which explains the opposite transformation: https://community.brain-map.org/t/how-to-transform-ccf-x-y-z-coordinates-into-stereotactic-coordinates/1858/3
# Warning: this is very approximate
# Warning: the X, Y, Z schematic at the top of the linked post is incorrect, scroll down for correct one.
def StereoToCCF(SC = np.array([1,1,1]), angle = -0.0873):
    # Stretch
    stretch = SC/np.array([1,0.9434,1])
    # Rotate
    rotate = np.array([(stretch[0] * math.cos(angle) - stretch[1] * math.sin(angle)),
                       (stretch[0] * math.sin(angle) + stretch[1] * math.cos(angle)),
                       stretch[2]])
    #Translate
    trans = rotate + np.array([5400, 440, 5700])
    return(trans)

def CCFToStereo(CCF = np.array([1,1,1]), angle = 0.0873):
    #Translate
    trans = CCF - np.array([5400, 440, 5700])
    # Rotate
    rotate = np.array([(trans[0] * math.cos(angle) - trans[1] * math.sin(angle)),
                       (trans[0] * math.sin(angle) + trans[1] * math.cos(angle)),
                       trans[2]])
    # Stretch
    stretch = rotate*np.array([1,0.9434,1])
    return(stretch)


In [15]:
def load_sorting_analzyers(project_path, mouse):
    # get sorting analyzer and unit locations
    day_paths = [f.path for f in os.scandir(f"{project_path}{mouse}/") if f.is_dir()]
    clusters = pd.DataFrame()
    for day_path in day_paths:
        print(day_path)
        sorting_analyzer_path = f"{day_path}/full/kilosort4/kilosort4_sa"
        try:
            if os.path.isdir(sorting_analyzer_path):
                sorting_analyzer = si.load_sorting_analyzer(sorting_analyzer_path)
                ulc = sorting_analyzer.get_extension("unit_locations")
                qms = sorting_analyzer.get_extension("quality_metrics")
                unit_locations = ulc.get_data(outputs="by_unit")
                quality_metrics = qms.get_data()
                quality_metrics["cluster_id"] = quality_metrics.index
                quality_metrics['unit_location_x'] = quality_metrics.index.map(lambda unit: unit_locations[unit][0])
                quality_metrics['unit_location_y'] = quality_metrics.index.map(lambda unit: unit_locations[unit][1])
                quality_metrics = quality_metrics[(quality_metrics["snr"] > 1) & 
                                                (quality_metrics["rp_contamination"] < 0.9)]
                clusters = pd.concat([clusters, quality_metrics], ignore_index=True)
        except:
            continue
    return clusters

In [16]:
def add_clusters(probe_locations_path_list, project_path, mouse):
    prob_locs_list = []
    for probe_locations_path in probe_locations_path_list:
        probe_locs = read_probe_mat(probe_locations_path)
        prob_locs = np.array([[probe_locs[0,0], probe_locs[0,1]], 
                              [probe_locs[2,0], probe_locs[2,1]],
                              [probe_locs[1,0], probe_locs[1,1]]])*10
        prob_locs_list.append(prob_locs)
    prob_locs_list = np.array(prob_locs_list)

    clusters_df = load_sorting_analzyers(project_path, mouse)
    # unit_location_x is ML
    # unit_location_y is DV
    # (0, 0) is the tip of the medial most shank and move +/+ in a lateral/dorsal direction
    # Add to scene
    return clusters_df

In [17]:
for Mouse in ['M25']:
    clusters_df = add_clusters(probe_locations_path_list=[f'/home/ubuntu/Elrond/probe_data/{Mouse}_probe_locations_1.mat',
                                                          f'/home/ubuntu/Elrond/probe_data/{Mouse}_probe_locations_2.mat',
                                                          f'/home/ubuntu/Elrond/probe_data/{Mouse}_probe_locations_3.mat',
                                                          f'/home/ubuntu/Elrond/probe_data/{Mouse}_probe_locations_4.mat'],
                                                          project_path="/mnt/datastore/Chris/Cohort12/derivatives/", 
                                                          mouse=Mouse)

[[999.57444032 994.45811187]
 [931.28725606 935.15175   ]
 [177.56466383 546.50895349]]
[[1003.83821648 1008.91178352]
 [ 903.11755666  923.50744334]
 [ 155.4260009   458.6989991 ]]
[[1014.70673974 1007.43157145]
 [ 872.98199803  891.61120299]
 [ 140.80904728  403.0475354 ]]
[[ 999.05829495 1013.46280478]
 [ 859.45691036  868.32502752]
 [ 119.11480048  310.36820985]]
/mnt/datastore/Chris/Cohort12/derivatives/M25/D19


/mnt/datastore/Chris/Cohort12/derivatives/M25/D15
/mnt/datastore/Chris/Cohort12/derivatives/M25/D21


/home/ubuntu/miniconda3/envs/elrond/lib/python3.11/site-packages/spikeinterface/core/base.py:1129: UserWarning: Versions are not the same. This might lead to compatibility errors. Using spikeinterface==0.102.0 is recommended
  warnings.warn(


/mnt/datastore/Chris/Cohort12/derivatives/M25/D16


/home/ubuntu/miniconda3/envs/elrond/lib/python3.11/site-packages/spikeinterface/core/base.py:1129: UserWarning: Versions are not the same. This might lead to compatibility errors. Using spikeinterface==0.102.0 is recommended
  warnings.warn(


...

/home/ubuntu/miniconda3/envs/elrond/lib/python3.11/site-packages/spikeinterface/core/base.py:1129: UserWarning: Versions are not the same. This might lead to compatibility errors. Using spikeinterface==0.102.0 is recommended
  warnings.warn(


/mnt/datastore/Chris/Cohort12/derivatives/M25/D8
/mnt/datastore/Chris/Cohort12/derivatives/M25/D3
/mnt/datastore/Chris/Cohort12/derivatives/M25/D12
/mnt/datastore/Chris/Cohort12/derivatives/M25/D29
/mnt/datastore/Chris/Cohort12/derivatives/M25/D25
/mnt/datastore/Chris/Cohort12/derivatives/M25/D13
/mnt/datastore/Chris/Cohort12/derivatives/M25/D26
/mnt/datastore/Chris/Cohort12/derivatives/M25/D18
/mnt/datastore/Chris/Cohort12/derivatives/M25/D14
/mnt/datastore/Chris/Cohort12/derivatives/M25/D7
/mnt/datastore/Chris/Cohort12/derivatives/M25/D2
/mnt/datastore/Chris/Cohort12/derivatives/M25/D5
/mnt/datastore/Chris/Cohort12/derivatives/M25/D32


/home/ubuntu/miniconda3/envs/elrond/lib/python3.11/site-packages/spikeinterface/core/base.py:1129: UserWarning: Versions are not the same. This might lead to compatibility errors. Using spikeinterface==0.102.0 is recommended
  warnings.warn(


/mnt/datastore/Chris/Cohort12/derivatives/M25/D27


/home/ubuntu/miniconda3/envs/elrond/lib/python3.11/site-packages/spikeinterface/core/base.py:1129: UserWarning: Versions are not the same. This might lead to compatibility errors. Using spikeinterface==0.102.0 is recommended
  warnings.warn(
